In [ ]:
import igraph as ig
import leidenalg as la
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)


In [ ]:
def run_leiden(G, quality, gamma, weights, seed):
    part_cls = la.RBConfigurationVertexPartition if quality.upper()=="RB" else la.CPMVertexPartition
    return la.find_partition(G, part_cls, seed=seed, n_iterations=-1,
                             resolution_parameter=gamma, weights=weights)

def quality_of_membership(G, quality, gamma, weights, membership):
    part_cls = la.RBConfigurationVertexPartition if quality.upper()=="RB" else la.CPMVertexPartition
    part = part_cls(G, initial_membership=membership,
                    resolution_parameter=gamma, weights=weights)
    return float(part.quality())

def summarize_membership(membership, node_names=None):
    n = len(membership)
    if node_names is None:
        node_names = list(range(n))
    df = pd.DataFrame({"node_id": range(n),
                       "node_name": node_names,
                       "community": membership})
    comm_sizes = (df.groupby("community").size()
                    .reset_index(name="n_nodes")
                    .sort_values("n_nodes", ascending=False)
                    .reset_index(drop=True))
    return df, comm_sizes

def relabel_consecutive(memb):
    uniq = sorted(set(memb))
    mp = {c:i for i,c in enumerate(uniq)}
    return [mp[c] for c in memb]

def inter_comm_pairs(G, membership, weights):
    uniq = sorted(set(membership))
    lab2idx = {c:i for i,c in enumerate(uniq)}
    memb_idx = np.array([lab2idx[c] for c in membership], dtype=np.int32)
    k = len(uniq)

    w_sum = [defaultdict(float) for _ in range(k)]
    w_attr = np.array(weights, dtype=float) if (weights is not None) else None

    for eid, e in enumerate(G.es):
        u, v = e.tuple
        cu, cv = memb_idx[u], memb_idx[v]
        w = float(w_attr[eid]) if w_attr is not None else 1.0
        if cu != cv:
            w_sum[cu][cv] += w
            w_sum[cv][cu] += w

    # index → label 로 변환
    idx2lab = {i:lab for i, lab in enumerate(uniq)}
    nbrs = {idx2lab[i]: sorted([(idx2lab[j], w) for j, w in w_sum[i].items()],
                               key=lambda x: x[1], reverse=True)
            for i in range(k)}
    return nbrs

def merge_pair_by_labels(memb, label_a, label_b):
    nm = [label_a if c==label_b else c for c in memb]
    return relabel_consecutive(nm)

# (gamma, seed) 탐색: max_size <= MAX_SIZE
def search_gamma_seed_maxcap(G, weights, quality, gammas, seeds, MAX_SIZE, node_names):
    records = []
    best = None  # {'gamma','seed','membership','quality','comm_sizes'}

    for g in gammas:
        for s in seeds:
            part = run_leiden(G, quality, g, weights, s)
            m = list(part.membership)
            _, cs = summarize_membership(m, node_names)
            mx = int(cs["n_nodes"].max())
            q  = quality_of_membership(G, quality, g, weights, m)
            records.append({"gamma": g, "seed": s, "max_size": mx, "k": int(cs.shape[0]), "quality": q})
            if mx <= MAX_SIZE:
                if (best is None) or (q > best["quality"]):
                    best = {"gamma": g, "seed": s, "membership": m, "quality": q, "comm_sizes": cs}

    # 실패 시: gamma 지수 상승으로 재시도
    if best is None:
        g0 = gammas[-1] if len(gammas)>0 else 0.2
        g_cap = 1e6
        g = max(g0, 1e-3)
        while g <= g_cap and best is None:
            g *= 1.5
            for s in seeds:
                part = run_leiden(G, quality, g, weights, s)
                m = list(part.membership)
                _, cs = summarize_membership(m, node_names)
                mx = int(cs["n_nodes"].max())
                q  = quality_of_membership(G, quality, g, weights, m)
                records.append({"gamma": g, "seed": s, "max_size": mx, "k": int(cs.shape[0]), "quality": q})
                if mx <= MAX_SIZE:
                    best = {"gamma": g, "seed": s, "membership": m, "quality": q, "comm_sizes": cs}
                    break

    return best, pd.DataFrame(records)

# 유사도 기반 병합(10~20 만족)
def pack_by_similarity(G, weights, membership, MIN_SIZE, MAX_SIZE, node_names):
    """
    - 입력 membership은 이미 max_size <= MAX_SIZE를 만족(=1단계 결과)
    - inter-community weight(유사도) 큰 이웃끼리 먼저 병합하되, 합친 크기가 MAX_SIZE 넘지 않도록
    - 작은 커뮤니티(<MIN_SIZE)를 우선 소거; 필요 시 여러 개 연쇄 병합
    - 그래도 불가하면 (연결 이웃이 없거나 모두 넘치는 경우) 임의 이웃 중에서 사이즈 제약 만족하는 후보로 병합
    """
    memb = relabel_consecutive(membership)
    while True:
        _, cs = summarize_membership(memb, node_names)
        if cs.empty:
            return memb, {"ok": False, "reason": "empty partition"}
        size_by = dict(zip(cs["community"], cs["n_nodes"]))

        # 종료 조건
        mn, mx = int(cs["n_nodes"].min()), int(cs["n_nodes"].max())
        if (mn >= MIN_SIZE) and (mx <= MAX_SIZE):
            return memb, {"ok": True}

        # 작은 커뮤니티 목록(오름차순)
        small = [int(c) for c, sz in size_by.items() if sz < MIN_SIZE]
        if not small:
            # 더 이상 작은 게 없는데 max>MAX_SIZE가 있을 수는 없음
            return memb, {"ok": (mx <= MAX_SIZE)}

        # 유사도 이웃 계산
        nbrs = inter_comm_pairs(G, memb, weights)

        progressed = False
        # 작은 것부터 처리
        small_sorted = sorted(small, key=lambda c: size_by[c])
        for c in small_sorted:
            sz_c = size_by[c]
            if sz_c >= MIN_SIZE:
                continue

            # 유사도 이웃(가중치 내림차순)에서 단일 병합으로 해결 가능한 후보
            for nb, w in nbrs.get(c, []):
                if nb == c: 
                    continue
                if size_by[nb] + sz_c <= MAX_SIZE:
                    memb = merge_pair_by_labels(memb, c, nb)  # nb를 c로 병합
                    progressed = True
                    break
            if progressed:
                break

            # 다중 병합: 이웃 여러 개를 더해 MIN_SIZE를 채우되 MAX_SIZE 넘지 않게
            acc_size = sz_c
            chosen = []
            for nb, w in nbrs.get(c, []):
                if nb == c: 
                    continue
                if acc_size >= MIN_SIZE:
                    break
                if size_by[nb] < MIN_SIZE and (acc_size + size_by[nb] <= MAX_SIZE):
                    chosen.append(nb)
                    acc_size += size_by[nb]
            if acc_size >= MIN_SIZE and chosen:
                # 선택된 nb들을 차례로 c에 병합
                for nb in chosen:
                    memb = merge_pair_by_labels(memb, c, nb)
                progressed = True
                break

            # 유사도 이웃이 없거나(단절) 모두 MAX 초과라면
            #    - 전체 커뮤니티 중에서 사이즈 제약을 지키는 임의 후보를 찾아 병합(유사도 제약 완화)
            candidates = [cc for cc, sz in size_by.items() if cc != c and (sz_c + sz) <= MAX_SIZE]
            if candidates:
                # 후보 중에서 nb와의 유사도(있으면) 큰 것, 없으면 사이즈가 가까운 것
                scored = []
                nbr_map = dict(nbrs.get(c, []))
                for cc in candidates:
                    score = nbr_map.get(cc, 0.0)  # 유사도가 없으면 0
                    scored.append((cc, score, abs((sz_c + size_by[cc]) - ((MIN_SIZE+MAX_SIZE)//2))))
                # 유사도 내림차순, 합친 사이즈가 중간값에 가까운 순
                scored.sort(key=lambda t: (t[1], -t[2]), reverse=True)
                nb = scored[0][0]
                memb = merge_pair_by_labels(memb, c, nb)
                progressed = True
                break

        if not progressed:
            # 더 이상 진행 불가(제약 충족 실패)
            return memb, {"ok": False, "reason": "no feasible merges under size constraints"}


In [ ]:
GRAPHML_PATH = "HR_HomG_drop005_onlySLremoved_0905.graphml"
WEIGHT_ATTR  = "weight"      # 없으면 None
QUALITY      = "RB"          # RB 기준
QUALITY_CPM = "CPM"
SEEDS        = [0,1,2,3,4]
GAMMAS       = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 0.2, 0.35, 0.5, 0.8, 1.2, 2.0, 3.0]

MIN_SIZE     = 10
MAX_SIZE     = 20

OUT_DIR = Path("results/leiden_C_plan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 그래프 로드
G = ig.Graph.Read_GraphML(GRAPHML_PATH)
if not G.is_directed():
    G = G.as_directed()
weights = G.es[WEIGHT_ATTR] if (WEIGHT_ATTR and (WEIGHT_ATTR in G.es.attributes())) else None
node_names = G.vs["id"] if "id" in G.vs.attributes() else list(range(G.vcount()))

# (gamma, seed) 탐색: max_size ≤ 20
best, eval_stage1 = search_gamma_seed_maxcap(
    G, weights, QUALITY_CPM, GAMMAS, SEEDS, MAX_SIZE, node_names
)
eval_stage1 = eval_stage1.sort_values(["max_size", "quality"], ascending=[True, False]).reset_index(drop=True)
display(eval_stage1.head(12))
eval_stage1.to_csv(OUT_DIR / f"C_eval_stage1_CPM_max{MAX_SIZE}.csv", index=False) # 중간에 RB를 CPM으로 바꿀 수도 있음. 

if best is None:
    raise RuntimeError("C-plan: no (gamma, seed) produced max_size ≤ MAX_SIZE; consider enlarging GAMMAS or escalation cap.")

best_gamma = float(best["gamma"])
best_seed  = int(best["seed"])
print(f"[C-Stage1] best_gamma={best_gamma}, best_seed={best_seed}, "
      f"k={int(best['comm_sizes'].shape[0])}, max_size={int(best['comm_sizes']['n_nodes'].max())}, "
      f"quality={best['quality']:.6f}")

# 선택된 (gamma, seed)로 재실행(재현성)
part0 = run_leiden(G, QUALITY_CPM, best_gamma, weights, best_seed)
m0 = list(part0.membership)
_, cs0 = summarize_membership(m0, node_names)
print(f"[C-Stage2] k={int(cs0.shape[0])}, min={int(cs0['n_nodes'].min())}, max={int(cs0['n_nodes'].max())}")

# 유사도(간선가중치) 기반 병합으로 10~20 충족
m_final, info = pack_by_similarity(
    G, weights, m0, MIN_SIZE, MAX_SIZE, node_names
)
_, cs_final = summarize_membership(m_final, node_names)
q_final = quality_of_membership(G, QUALITY_CPM, best_gamma, weights, m_final)

ok_flag = bool(info.get("ok", False))
print(f"[C-Stage3] ok={ok_flag}, reason='{info.get('reason','')}', "
      f"k={int(cs_final.shape[0])}, min={int(cs_final['n_nodes'].min())}, max={int(cs_final['n_nodes'].max())}, "
      f"quality={q_final:.6f}")

eval_final = pd.DataFrame([{
    "gamma": best_gamma, "seed": best_seed, "ok": int(ok_flag),
    "k": int(cs_final.shape[0]),
    "min_size": int(cs_final["n_nodes"].min()),
    "max_size": int(cs_final["n_nodes"].max()),
    "quality": q_final,
    "reason": info.get("reason", "")
}])
eval_final.to_csv(OUT_DIR / f"C_eval_final_CPM_min{MIN_SIZE}_max{MAX_SIZE}.csv", index=False)

df_membership = pd.DataFrame({
    "node_id": range(G.vcount()),
    "node_name": node_names,
    "community": m_final
})
df_membership.to_csv(OUT_DIR / f"C_membership_CPM_g{best_gamma:.6g}_s{best_seed}_min{MIN_SIZE}_max{MAX_SIZE}.csv", index=False)
cs_final.to_csv(OUT_DIR / f"C_comm_sizes_CPM_g{best_gamma:.6g}_s{best_seed}_min{MIN_SIZE}_max{MAX_SIZE}.csv", index=False)

print("saved:", OUT_DIR)


In [ ]:
GRAPHML_IN    = Path("HR_HomG_drop005_onlySLremoved_0905.graphml")  # 원본 GraphML
MEMBERSHIP_CSV= Path("results/C_membership_CPM_g0.001_s1_min10_max20.csv")  # C 결과 CSV
GRAPHML_OUT   = Path("results/HR_HomG_drop005_LeidenClu_ewX.graphml")

G = ig.Graph.Read_GraphML(str(GRAPHML_IN))
if not G.is_directed():
    G = G.as_directed()
dfm = pd.read_csv(MEMBERSHIP_CSV)

if "id" not in G.vs.attributes():
    raise RuntimeError(f"그래프 vertex 속성에 'id'가 없습니다. 현재: {G.vs.attributes()}")
for col in ("node_name", "community"):
    if col not in dfm.columns:
        raise ValueError(f"CSV에 '{col}' 컬럼이 없습니다. 현재: {list(dfm.columns)}")

# 'id'(그래프) <-> 'node_name'(CSV) 매핑
graph_ids = [str(x) for x in G.vs["id"]]
comm_map  = dict(zip(dfm["node_name"].astype(str), dfm["community"].astype(int)))

missing = [gid for gid in graph_ids if gid not in comm_map]
if missing:
    preview = ", ".join(missing[:5])
    raise RuntimeError(f"CSV에 없는 그래프 id가 {len(missing)}개 있습니다. 예: {preview}")

# 커뮤니티 속성 부여
G.vs["community"] = [int(comm_map[gid]) for gid in graph_ids]

# 모든 엣지 weight=1
G.es["weight"] = [1.0] * G.ecount()

G.vs["label"] = graph_ids               # 라벨로 사용될 텍스트 이름
G.vs["name"]  = graph_ids               # 호환용(igraph의 vertex 'name' 관례)


GRAPHML_OUT.parent.mkdir(parents=True, exist_ok=True)
G.write_graphml(str(GRAPHML_OUT))
print(f"[완료] community 매핑 + weight=1 + label/name 설정: {GRAPHML_OUT}")
